# 04 — Data & evaluation: the loss function

**What you'll learn**

- Turning raw data into `dspy.Example` splits (train / dev / test) — and why the hygiene matters here as much as in ML.
- Writing a **metric** — DSPy's loss function — and the meaning of its mysterious third argument, `trace`.
- `dspy.Evaluate`: the parallel eval loop, and reading its `EvaluationResult`.
- Error analysis that turns a score into a *diagnosis* (which the optimizers will then treat).

*Time: ~25 min. Cost: ~$0.05. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, and the second line finds or starts the MLflow UI — click the printed link to browse them. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()
obs.mlflow_ui()            # prints the link where the traces appear

Same code as notebook 03 — skim and run. (Every notebook in this course rebuilds its world from scratch, so you can open any of them cold.)

In [ ]:
# === boilerplate: aldervane micro-world — same code as notebook 03 ===
import json, math, re
from collections import Counter
from pathlib import Path

CORPUS = json.loads(Path("data/corpus.json").read_text())

STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = {p["id"]: set(tokenize(p["title"] + " " + p["text"])) for p in CORPUS}
IDF = Counter()
for tokens in DOC_TOKENS.values():
    IDF.update(tokens)
IDF = {t: math.log(len(CORPUS) / df) for t, df in IDF.items()}

def search(query, k=2):
    """Return the k corpus passages that best match the query (keyword + IDF)."""
    q = set(tokenize(query))
    ranked = sorted(
        CORPUS,
        key=lambda p: (-sum(IDF.get(t, 0.0) for t in q & DOC_TOKENS[p["id"]]), p["id"]),
    )
    return ranked[:k]

def as_context(passages):
    return [f"{p['title']}: {p['text']}" for p in passages]

class HopQA(dspy.Module):
    """Two-hop retrieval QA over the Aldervane corpus."""

    def __init__(self, k=2):
        super().__init__()
        self.k = k
        self.hop1 = dspy.Predict("question -> search_query")
        self.hop2 = dspy.Predict("question, context: list[str] -> search_query")
        self.answer = dspy.ChainOfThought("question, context: list[str] -> answer")

    def forward(self, question):
        q1 = self.hop1(question=question).search_query
        first = search(q1, k=self.k)
        q2 = self.hop2(question=question, context=as_context(first)).search_query
        second = search(q2, k=self.k)
        passages = {p["id"]: p for p in first + second}      # merge + dedupe
        pred = self.answer(question=question, context=as_context(list(passages.values())))
        return dspy.Prediction(
            answer=pred.answer,
            reasoning=pred.reasoning,
            queries=[q1, q2],
            retrieved_ids=sorted(passages),
        )

## From JSON to Examples

You cannot improve what you cannot measure, and you cannot measure without data. `data/qa.json` holds 48 hand-written questions about Aldervane, pre-split:

```
train (24)   what optimizers may learn from
dev   (16)   what we measure progress on, as often as we like
test  ( 8)   sealed. opened exactly twice in this course: notebooks 07 and 10
```

The splits obey the same law as in all of ML: **the thing you tune on leaks into the thing you tune.** Optimizers will consume `train`; we will steer by `dev`; `test` exists so that, at the very end, one honest number remains possible. Each row also records `gold_ids` — which passages contain the evidence — and `hops`. Labels like these never reach the program (only `question` is an input); they exist for the *metric's* benefit.

In [ ]:
QA = json.loads(Path("data/qa.json").read_text())

def to_examples(rows):
    return [dspy.Example(**row).with_inputs("question") for row in rows]

train, dev, test = (to_examples(QA[s]) for s in ("train", "dev", "test"))

print(len(train), len(dev), len(test))
ex = train[10]
print(ex.inputs())     # what the program sees
print(ex.labels())     # what only the metric sees

> **What you should see (deterministic):** `24 16 8`, then the input view (`question` only) and the label view (`answer`, `hops`, `gold_ids`) of one training example.

## The metric — a loss function you write in plain Python

A DSPy metric is just a function `metric(example, prediction, trace=None) -> score`. It encodes what "good" means for *your* task; nothing about it is built-in or sacred.

Ours will be **normalized exact match**: strict about content, forgiving about case, punctuation, and articles. Strictness is a deliberate choice — the model tends to answer in sentences ("The tidewriter, a brass instrument...") while our gold answers are minimal ("tidewriter"). A strict metric makes verbosity *cost points*, which creates exactly the pressure the optimizers will later relieve. A metric is not just a measurement; it is the objective your program will be optimized toward. Design it like you mean it.

In [ ]:
def norm(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em_metric(example, pred, trace=None):
    """1.0 if the predicted answer exactly matches the gold answer (after normalization)."""
    return norm(pred.answer) == norm(example.answer)

About that third parameter. During ordinary evaluation, `trace` is `None` — return a score (ours is a bool; True counts as 1.0). But during *optimization*, DSPy will sometimes call your metric with `trace` set to the program's full execution record, asking a different question: not "how good is this?" but **"is this execution good enough to learn from?"** — expecting a strict yes/no. Notebook 05 reveals who passes `trace` and why it changes the metric's job. For exact match the two answers coincide, so this function is already both.

In [ ]:
program = HopQA()
pred = program(question=dev[0].question)
print("Q:   ", dev[0].question)
print("gold:", dev[0].answer, "| predicted:", pred.answer)
print("metric says:", em_metric(dev[0], pred))

> **What you should see:** one live example of the metric judging one prediction. If the model answered with a sentence containing the right entity, watch the metric say `False` anyway — that is the strictness, working as intended.

## `dspy.Evaluate` — the eval loop, industrialized

You could write the loop yourself: run 16 questions, collect scores, average. `dspy.Evaluate` does that with threading, progress, an error budget, and a results table. Note: its arguments are keyword-only.

In [ ]:
evaluate = dspy.Evaluate(
    devset=dev,
    metric=em_metric,
    num_threads=8,
    display_progress=True,
    display_table=5,
)

result = evaluate(program)
result.score

> **What you should see:** a progress bar over 16 examples, a rendered table of the first 5 (question, gold answer, prediction, score), and a final score — **a percentage, typically landing anywhere from ~25 to ~55 for this untrained program**. Small dev set, so each question is worth 6.25 points; don't read precision into single-digit differences.
>
> The return value is an `EvaluationResult`: `.score` (the percentage) and `.results` (a list of `(example, prediction, score)` triples — the raw material for error analysis, next).

**The leaderboard.** From here through notebook 07 we keep score in one table. Copy your number in; the "expect" column is a coarse range, not a promise — tiny dev sets are noisy.

| program | dev EM (16 q) | expect | notebook |
|---|---|---|---|
| `HopQA()` zero-shot | *your number* | ~25–55 | 04 |

## From score to diagnosis

A single number tells you *that* the program fails, never *why*. But our `forward` returns diagnostics (`queries`, `retrieved_ids`) precisely for this moment — and each example carries `gold_ids`. Let's cross-examine the failures:

In [ ]:
failures = [(ex, pred) for ex, pred, score in result.results if not score]
print(f"{len(failures)} failures / {len(dev)}\n")

for ex, pred in failures[:3]:
    missing = set(ex.gold_ids) - set(pred.retrieved_ids)
    print("Q:        ", ex.question)
    print("gold:     ", ex.answer, " | predicted:", pred.answer)
    print("queries:  ", pred.queries)
    print("retrieved:", pred.retrieved_ids, " gold:", ex.gold_ids,
          " MISSING:", sorted(missing) if missing else "none")
    print("-" * 70)

> **What you should see:** the failures sort themselves into two families.
>
> 1. **Retrieval failures** (`MISSING` non-empty, mostly on 2-hop questions): look at `queries[1]` — the second query tends to restate the question instead of naming the *bridge entity* found in hop 1, so the second search finds nothing new. The evidence for the answer was never on the table.
> 2. **Answering failures** (`MISSING: none`): the right passages were retrieved, but the answer is verbose, or names the wrong lookalike entity (Aldervane is full of traps — two Venns, two Rasks, two markets, two foundries).
>
> Keep this taxonomy. Notebook 05's optimizer will visibly treat *both* diseases, and in notebook 07 we will hand this very diagnosis, as text, to an optimizer that reads it.

## What did that cost?

Optimizers multiply evaluation calls, so cost awareness starts now. DSPy meters usage per prediction if you ask:

In [ ]:
dspy.configure(track_usage=True)

pred = program(question="Who signs every harbor ledger on Halewick Isle?")
pred.get_lm_usage()

> **What you should see:** a dict keyed by model name with prompt/completion token counts covering all three LM calls of this one program run. **If you see `{}`:** the cache answered (cached calls consume zero tokens — this line is only fresh the first time). Change the question string slightly to force a real call.
>
> Mental arithmetic for later: one `HopQA` call ≈ 3 LM calls; a 16-example eval ≈ 48; an optimizer that tries 10 candidate programs ≈ 500. The disk cache is what makes re-*evaluating* cheap; only genuinely new prompts cost money.

## The wider evaluation toolbox (a guided glance)

Everything above was hand-rolled so you would see the moving parts. Know what ships in the box:

- **Ready-made metrics:** `dspy.evaluate.answer_exact_match`, `answer_passage_match`; token-level `F1` lives in `dspy.evaluate.metrics`.
- **LLM-as-judge metrics:** `dspy.evaluate.SemanticF1()` and `CompleteAndGrounded()` score *meaning* rather than string equality. Pause on what these are: they are themselves `dspy.Module`s — **your loss function can be a DSPy program**, with its own signature, its own predictors... and yes, in principle, its own optimizer. (We stay with exact match: on this corpus the gold answers are short and unambiguous, and a deterministic metric keeps the optimizer notebooks legible.)
- **Datasets:** `dspy.datasets` ships `HotPotQA` (our capstone), `MATH`, `Colors`; `dspy.datasets.DataLoader` ingests CSV/JSON/pandas/HuggingFace (`from_csv`, `from_json`, `from_huggingface`, plus `train_test_split`, `sample`). One trap: `GSM8K` is *not* exported at the package top — it needs the deep import `from dspy.datasets.gsm8k import GSM8K, gsm8k_metric`.

## Recap

| Concept | One-liner |
|---|---|
| splits | train = optimizer food, dev = steering, test = sealed until 07/10 |
| labels | fields the program never sees; the metric sees everything |
| metric | `f(example, pred, trace=None) -> score`; strictness is leverage for optimizers |
| `trace` | `None` → "score it"; set → "is this worth learning from?" (notebook 05) |
| `dspy.Evaluate` | keyword-only; threaded; returns `EvaluationResult` (`.score`, `.results`) |
| error analysis | diagnostics in your `Prediction` + labels in your `Example` = diagnosis |
| `track_usage` | `dspy.configure(track_usage=True)` + `pred.get_lm_usage()` |

## Exercises

1. Write a token-level F1 metric (partial credit: overlap of `norm(...)` word sets). Re-evaluate. Which examples change verdict, and do you *agree* with the partial credit?
2. Write `retrieval_recall(example, pred, trace=None)` = fraction of `example.gold_ids` present in `pred.retrieved_ids`, and evaluate it. You now have per-stage diagnostics: is the pipeline mostly failing at retrieval or at answering? (File the answer away — it becomes GEPA feedback in notebook 07.)
3. Re-run the evaluation with `num_threads=1` and feel the difference. Then reflect: what made the threaded rerun near-instant? (Two separate mechanisms are involved.)

**Next up:** the first real optimizer. The program runs its own trainset, keeps the executions your metric blesses, and turns them into few-shot demos — per predictor. The bridge-query bug from today's diagnosis is about to fix itself.